# FP-Growth — Khai Phá Luật Kết Hợp (Association Rules)

## Nguồn dữ liệu
Notebook này đọc từ **Data Mart** được trích xuất từ **Data Warehouse (DuckDB)**:
- Mart: `data/warehouse/marts/mart_basket.csv`
- DW: `data/warehouse/instacart_dw.duckdb` (Snowflake Schema)

## Luồng dữ liệu
```
data/raws/*.csv
    → [ETL: extract.py → transform.py → load.py]
    → data/warehouse/instacart_dw.duckdb  (Snowflake Schema)
    → [Data Mart: mart_basket.py]
    → data/warehouse/marts/mart_basket.csv
    → Notebook này (FP-Growth)
```

In [ ]:
# ── Thư viện ──────────────────────────────────────────────────────────────
import sys
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

# Đảm bảo import được src/
sys.path.insert(0, '..')

MART_PATH = '../data/warehouse/marts/mart_basket.csv'
TOP_N_PRODUCTS = 1000   # Chỉ dùng 1000 sản phẩm phổ biến nhất để FP-Growth chạy nhanh
MIN_SUPPORT    = 0.001  # Ngưỡng support tối thiểu
MIN_LIFT       = 1.0    # Ngưỡng lift tối thiểu

print('Cấu hình:')
print(f'  Mart path   : {MART_PATH}')
print(f'  Top products: {TOP_N_PRODUCTS}')
print(f'  Min support : {MIN_SUPPORT}')
print(f'  Min lift    : {MIN_LIFT}')

In [ ]:
# ── Đọc từ Data Mart (trích xuất từ Data Warehouse) ──────────────────────
print('Đọc dữ liệu từ Data Mart...')
df = pd.read_csv(MART_PATH)

print(f'\nData Mart — mart_basket.csv:')
print(f'  Tổng dòng       : {len(df):,}')
print(f'  Unique orders   : {df["order_id"].nunique():,}')
print(f'  Unique products : {df["product_name"].nunique():,}')
print(f'  Unique aisles   : {df["aisle_name"].nunique():,}')
print(f'  Unique depts    : {df["department_name"].nunique():,}')
print(f'\nMẫu dữ liệu:')
print(df.head(5).to_string(index=False))

In [ ]:
# ── Lọc Top N sản phẩm phổ biến nhất ─────────────────────────────────────
# Lý do: FP-Growth với 50K sản phẩm sẽ tốn nhiều RAM
# → Giữ lại 1000 sản phẩm mua nhiều nhất vẫn đủ đại diện

top_products = df['product_name'].value_counts().nlargest(TOP_N_PRODUCTS).index
df_filtered = df[df['product_name'].isin(top_products)].copy()

print(f'Sau khi lọc top {TOP_N_PRODUCTS} sản phẩm:')
print(f'  Dòng còn lại  : {len(df_filtered):,}')
print(f'  Unique orders : {df_filtered["order_id"].nunique():,}')
print(f'  Tỷ lệ giữ lại : {len(df_filtered)/len(df)*100:.1f}%')

In [ ]:
# ── Chuyển về dạng Transaction List ──────────────────────────────────────
# Mỗi order_id → 1 transaction (list các sản phẩm)

transactions = (
    df_filtered
    .groupby('order_id')['product_name']
    .apply(list)
    .tolist()
)

print(f'Tổng transactions : {len(transactions):,}')
print(f'Ví dụ transaction đầu tiên: {transactions[0][:5]}...')

In [ ]:
# ── Encode → One-Hot Sparse Matrix ────────────────────────────────────────
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions, sparse=True)
basket_sets = pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

print(f'Transaction Matrix: {basket_sets.shape[0]:,} orders × {basket_sets.shape[1]:,} products')

In [ ]:
# ── Chạy FP-Growth → Tìm Frequent Itemsets ───────────────────────────────
import time

t0 = time.time()
frequent_itemsets = fpgrowth(basket_sets, min_support=MIN_SUPPORT, use_colnames=True)
t_fpgrowth = time.time() - t0

print(f'FP-Growth hoàn tất trong {t_fpgrowth:.2f}s')
print(f'Tìm được {len(frequent_itemsets):,} frequent itemsets')
print(f'\nTop 10 itemsets phổ biến nhất:')
print(
    frequent_itemsets
    .sort_values('support', ascending=False)
    .head(10)
    .to_string(index=False)
)

In [ ]:
# ── Sinh Association Rules ─────────────────────────────────────────────────
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=MIN_LIFT
)
rules = rules.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print(f'Tổng số luật kết hợp: {len(rules):,}')
print(f'\nTop 5 luật mạnh nhất (theo Lift):')
print(
    rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    .head(5)
    .to_string(index=False)
)

In [ ]:
# ── Hàm dự đoán sản phẩm ──────────────────────────────────────────────────
def predict_next_items(current_basket, association_rules_df, top_n=5):
    """
    Dự đoán sản phẩm tiềm năng từ giỏ hàng hiện tại.
    - current_basket     : list tên sản phẩm khách đang chọn
    - association_rules_df: DataFrame chứa các luật đã khai phá
    - top_n              : số sản phẩm gợi ý tối đa
    """
    basket_set = set(current_basket)

    # Lọc các luật có antecedents ⊆ giỏ hàng hiện tại
    matched = association_rules_df[
        association_rules_df['antecedents'].apply(lambda x: x.issubset(basket_set))
    ]

    # Sắp xếp theo Lift → Confidence (luật mạnh nhất lên đầu)
    matched = matched.sort_values(
        by=['lift', 'confidence'],
        ascending=[False, False]
    )

    # Thu thập consequents, loại bỏ sản phẩm đã trong giỏ
    suggested = []
    for item_set in matched['consequents']:
        for item in item_set:
            if item not in basket_set and item not in suggested:
                suggested.append(item)
                if len(suggested) == top_n:
                    return suggested
    return suggested


# ── Demo ──────────────────────────────────────────────────────────────────
test_baskets = [
    ['Banana'],
    ['Organic Baby Spinach', 'Organic Hass Avocado'],
    ['Strawberries', 'Raspberries'],
]

print('=== KẾT QUẢ GỢI Ý SẢN PHẨM (FP-Growth từ Data Mart) ===')
for basket in test_baskets:
    preds = predict_next_items(basket, rules, top_n=5)
    print(f'\n  Giỏ hàng   : {basket}')
    print(f'  Gợi ý A→B  : {preds}')

In [ ]:
# ── Phân tích luật theo Department (nhờ Data Mart có department_name) ─────
# Đây là lợi thế khi dùng DW: có thêm context từ dimension tables

product_dept = df.drop_duplicates('product_name').set_index('product_name')['department_name']

def get_dept(itemset):
    depts = [product_dept.get(p, 'unknown') for p in itemset]
    return list(set(depts))

rules_sample = rules.head(20).copy()
rules_sample['ant_dept'] = rules_sample['antecedents'].apply(get_dept)
rules_sample['con_dept'] = rules_sample['consequents'].apply(get_dept)

print('Top 10 luật mạnh nhất kèm thông tin phòng hàng (từ DW dimension):')
print(
    rules_sample[['antecedents', 'consequents', 'lift', 'ant_dept', 'con_dept']]
    .head(10)
    .to_string(index=False)
)